# Stream Model To CPU

In this notebook we will demonstrate how to read tensors using the RunAI Model Streamer to the CPU memory and perform computation on their data.

There are not requirements for this notebook, beside running on a Linux machine.

## Preperation
We will start by downloading an example `.safetensors` file. Feel free to use your own.

In [1]:
import subprocess

url = 'https://huggingface.co/vidore/colpali/resolve/main/adapter_model.safetensors?download=true'
local_filename = 'model.safetensors'

wget_command = ['wget', '--content-disposition', url, '-O', local_filename]
subprocess.run(wget_command, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)

## Streaming

To load the tensors from the file we need to create `SafetensorsStreamer` instance, perform the request, and start iterating the tensors.

One important thing to notice is that we perform `.detach().clone()` to have a copy instance of the tensor, as the buffer the streamer is reading into is not guarenteed to keep the tensor data (For example - when we reuse the buffer for other tensors)

In [2]:
from runai_model_streamer import SafetensorsStreamer

file_path = "model.safetensors"

def heavy_workload(tensor):
    # Perform heavy computation with the tensor
    # The computation is occured during the reading
    # of the rest of the tensors from the storage
    return

with SafetensorsStreamer() as streamer:
    streamer.stream_file(file_path)
    for name, tensor in streamer.get_tensors():
        cpu_tensor = tensor.detach().clone()
        heavy_workload(cpu_tensor)

Read throughput is 4.00 KB per second 
[RunAI Streamer] CPU Buffer size: 74.9 MiB for file: model.safetensors
Read throughput is 39.25 MB per second 
Read throughput is 197.45 MB per second 
[RunAI Streamer] Overall time to stream 74.9 MiB of all files: 0.41s, 184.7 MiB/s


A heavy workload can be running on each tensor in the moment the tensor is yielded - in parallel to the CPP threads that continue reading from the storage. 